# OPUS-DSD on Google Colab

<a href="https://colab.research.google.com/github/ts387/opusDSD/blob/main/colab/OPUS_DSD_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[OPUS-DSD](https://github.com/alncat/opusDSD) learns a latent space of structural heterogeneity from cryo-EM particle images that were aligned against a consensus model, and can render a 3D map for any point of that space. OPUS-DSD2 adds multi-body dynamics. This notebook installs it on a Colab GPU and walks through the whole workflow.

| Step | What it does |
|---|---|
| **1 · Setup** | Check the GPU, install OPUS-DSD, choose a project folder (Google Drive recommended) |
| **2 · Data** | **2A** generate a small synthetic demo dataset, **or** **2B** point to your own RELION `.star` file |
| **3 · Prepare** | Convert the STAR file to pose/CTF pickles and build a solvent mask; optionally define rigid bodies for OPUS-DSD2 |
| **4 · Train** | Run `dsd train_multi` in the background and watch it live; resumes after a disconnect |
| **5 · Analyze** | PCA, UMAP and k-means clustering of the learned latent space |
| **6 · Volumes** | Maps at the k-means centres and along principal components, with a 3D viewer |
| **7 · Export** | Per-cluster STAR files for RELION, and a zip of the results |

**How to use it.** Run the cells from top to bottom (Shift + Enter). Settings live in the form on the right of each cell; the defaults run the demo end to end. Double-click a form to see the code behind it.

**If Colab disconnects** (eventually it will): reconnect, re-run the three cells of step 1 (they restore your session from the project folder) and carry on. If you used 2B with *copy to local disk*, re-run 2B as well. Training continues from its last checkpoint when you re-run 4.1.

**What to expect.** The demo is sized for a free T4. A real dataset of tens of thousands of particles takes many hours per epoch on one T4 (the authors quote about 2 h per epoch for 20k particles on four V100s), so try a subset first (2B can draw one), and consider Colab Pro for an A100/L4 GPU and longer sessions. Checkpoints go to Drive, so a long training run can span several sessions.

## 1 · Setup

In [ ]:
#@title 1.1 · Check the GPU { display-mode: "form" }
import torch
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {p.name}, {p.total_memory / 2**30:.1f} GB')
    print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')
else:
    print('No GPU found. Training needs one: Runtime > Change runtime type > T4 GPU (or better), then run '
          'this cell again.\nPreparing data and analysing results also work without a GPU.')

In [ ]:
#@title 1.2 · Install OPUS-DSD (≈1 min) { display-mode: "form" }
#@markdown Where to get the code from. The default is the repository this notebook belongs to.
REPO_URL = "https://github.com/ts387/opusDSD"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

import os, sys, shutil, subprocess

REPO_DIR = '/content/opusDSD'

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f'failed: {cmd}')
    return r.stdout

if os.path.isdir(f'{REPO_DIR}/.git'):
    _run(f'git -C {REPO_DIR} remote set-url origin {REPO_URL} && '
         f'git -C {REPO_DIR} fetch -q --depth 1 origin {BRANCH} && git -C {REPO_DIR} checkout -q FETCH_HEAD')
else:
    _run(f'git clone -q --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}')
print('Code:', _run(f'git -C {REPO_DIR} log -1 --format="%h %s (%cd)" --date=short').strip())

# PyTorch, NumPy, SciPy, scikit-learn, pandas, matplotlib, seaborn, plotly and scikit-image ship with
# Colab. Installing the pinned requirements*.txt would downgrade them, so only add what is missing.
_run(f'{sys.executable} -m pip install -q healpy monty umap-learn')
_run(f'{sys.executable} -m pip install -q --no-deps -e {REPO_DIR}')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
for tool in ('dsd', 'dsdsh'):
    assert shutil.which(tool), f'{tool} was not installed'
    _run(f'{tool} -h')
print('Installed: the dsd and dsdsh commands are ready.')

In [ ]:
#@title 1.3 · Project folder, session and helpers { display-mode: "form" }
#@markdown Everything the notebook produces goes into one project folder. Keep it on **Google Drive** so checkpoints and results survive a disconnect. Re-running this cell restores the session saved there.
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
PROJECT_NAME = "opusdsd_project"  #@param {type:"string"}

import os, sys, re, json, glob, time, types, pickle, shlex, shutil, signal, base64, subprocess
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output

IN_COLAB = 'google.colab' in sys.modules
REPO_DIR = '/content/opusDSD'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
try:
    from cryodrgn import mrc as cmrc, starfile as cstar
except ImportError:
    raise RuntimeError('OPUS-DSD is not installed in this runtime: run cell 1.2 first.')

if not re.fullmatch(r'[\w.-]+', PROJECT_NAME):
    raise ValueError('PROJECT_NAME may only contain letters, digits, "_", "-" and "."')
if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    PROJECT_DIR = f'/content/drive/MyDrive/{PROJECT_NAME}'
else:
    PROJECT_DIR = f'/content/{PROJECT_NAME}'
LOCAL_DIR = '/content/opusdsd_local'  # fast disk of the VM, wiped when the runtime is recycled
for _d in (PROJECT_DIR, LOCAL_DIR):
    os.makedirs(_d, exist_ok=True)

SESSION_FILE = f'{PROJECT_DIR}/session.json'
S = json.load(open(SESSION_FILE)) if os.path.exists(SESSION_FILE) else {}
S.setdefault('runs', {})
_TRAIN_PROCS = globals().get('_TRAIN_PROCS', {})
q = shlex.quote

def save_session():
    with open(SESSION_FILE, 'w') as f:
        json.dump(S, f, indent=1)

def need(key, step):
    if not S.get(key):
        raise RuntimeError(f'Run step {step} first.')
    return S[key]

def sh(cmd, quiet=False):
    """Run a shell command with live output and raise if it fails. dsdsh exits with 0 even when the
    command it wraps crashed, so a Python traceback in the output counts as a failure as well."""
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    chunks = []
    while True:
        chunk = p.stdout.read1(1 << 16)
        if not chunk:
            break
        chunks.append(chunk.decode(errors='replace'))
        if not quiet:
            sys.stdout.write(chunks[-1]); sys.stdout.flush()
    p.wait()
    out = ''.join(chunks)
    if p.returncode != 0 or 'Traceback (most recent call last)' in out:
        if quiet:
            print(out[-6000:])
        raise RuntimeError(f'Command failed (exit code {p.returncode}), see the output above.')
    return out

# ---- maps ------------------------------------------------------------------------------------
def read_mrc(path):
    vol, hdr = cmrc.parse_mrc(path)
    return np.asarray(vol, dtype=np.float32), float(hdr.get_apix())

def write_mrc(path, vol, apix):
    cmrc.write(path, np.ascontiguousarray(vol, dtype=np.float32), Apix=apix, is_vol=True)

def fit_to_box(vol, vol_apix, D, apix):
    """resample a map to the pixel size of the particles and pad or crop it to a D^3 box"""
    from scipy.ndimage import zoom
    if abs(vol_apix - apix) > 1e-3 * apix:
        vol = zoom(vol, vol_apix / apix, order=1)
    out = np.zeros((D, D, D), np.float32)
    src, dst = [], []
    for n in vol.shape:
        s = abs(n - D) // 2
        src.append(slice(s, s + D) if n >= D else slice(0, n))
        dst.append(slice(0, D) if n >= D else slice(s, s + n))
    out[tuple(dst)] = vol[tuple(src)]
    return out

def iso_level(vol):
    from skimage.filters import threshold_otsu
    return float(threshold_otsu(vol))

def mrc_series(folder):
    files = glob.glob(f'{folder}/reference*.mrc')
    return sorted(files, key=lambda f: int(re.findall(r'reference(\d+)\.mrc', f)[0]))

# ---- figures ---------------------------------------------------------------------------------
def _views(vol, kind):
    if kind == 'slice':
        c = [n // 2 for n in vol.shape]
        return [vol[c[0]], vol[:, c[1]], vol[:, :, c[2]]]
    return [vol.sum(axis=a) for a in range(3)]

def show_views(vols, titles, kind='proj', masks=None, size=2.4):
    """one row per map, viewed down z, y and x; kind is 'proj' (projections) or 'slice' (central slices)"""
    fig, axes = plt.subplots(len(vols), 3, figsize=(3 * size, len(vols) * size), squeeze=False)
    for i, vol in enumerate(vols):
        for j, im in enumerate(_views(vol, kind)):
            ax = axes[i, j]
            ax.imshow(im, cmap='gray', origin='lower')
            for k, m in enumerate(masks[i] if masks is not None and masks[i] is not None else []):
                outline = _views(m, 'slice')[j] if kind == 'slice' else m.max(axis=j)
                ax.contour(outline, levels=[0.5], colors=[plt.cm.tab10(3 + k)], linewidths=1)
            ax.set_xticks([]); ax.set_yticks([])
            if i == 0:
                ax.set_title(f'{"central slice" if kind == "slice" else "projection"} along {"zyx"[j]}', fontsize=9)
        axes[i, 0].set_ylabel(titles[i], fontsize=9)
    plt.tight_layout(); plt.show()

def show_gallery(images, titles, ncols=8, size=1.5):
    n = len(images); ncols = min(n, ncols); nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * (size + 0.25)), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, im, t in zip(axes.flat, images, titles):
        ax.imshow(im, cmap='gray', origin='lower'); ax.set_title(t, fontsize=8)
    plt.tight_layout(); plt.show()

def show_pngs(paths, width=380):
    html = ''
    for p in paths:
        if os.path.exists(p):
            b64 = base64.b64encode(open(p, 'rb').read()).decode()
            html += (f'<figure style="display:inline-block;margin:4px;text-align:center">'
                     f'<img src="data:image/png;base64,{b64}" width="{width}">'
                     f'<figcaption style="font-size:11px">{os.path.relpath(p, PROJECT_DIR)}</figcaption></figure>')
    display(HTML(html))

def show_3d(vols, titles, apix=1.0, level=0, max_box=56):
    """interactive isosurfaces of maps of the same size; with several maps a slider switches between them"""
    import plotly.graph_objects as go
    from skimage.measure import marching_cubes
    from scipy.ndimage import zoom
    f = min(1.0, max_box / max(vols[0].shape))
    small = [zoom(v, f, order=1) if f < 1 else v for v in vols]
    lv = level or iso_level(np.stack(small))  # one level for the whole series, so that maps compare
    traces, names = [], []
    for v, title in zip(small, titles):
        try:
            verts, faces, _, _ = marching_cubes(v, lv, spacing=(apix / f,) * 3)
        except (ValueError, RuntimeError):
            print(f'{title}: nothing above the iso level {lv:.3g}')
            continue
        verts, faces = verts.astype(np.float32), faces.astype(np.int32)
        traces.append(go.Mesh3d(x=verts[:, 2], y=verts[:, 1], z=verts[:, 0], i=faces[:, 0], j=faces[:, 1],
                                k=faces[:, 2], color='#9db4d6', visible=not traces, hoverinfo='skip',
                                lighting=dict(ambient=0.35, diffuse=0.8, specular=0.2, roughness=0.6),
                                lightposition=dict(x=1000, y=2000, z=1500)))
        names.append(f'{title}   (iso level {lv:.3g})')
    if not traces:
        return
    fig = go.Figure(traces)
    if len(traces) > 1:
        steps = [dict(method='update', label=str(i), args=[{'visible': [k == i for k in range(len(traces))]},
                                                           {'title.text': names[i]}]) for i in range(len(traces))]
        fig.update_layout(sliders=[dict(active=0, steps=steps, currentvalue=dict(visible=False), pad=dict(t=5))])
    hidden = dict(visible=False)
    fig.update_layout(title=names[0], height=520, margin=dict(l=0, r=0, t=40, b=0),
                      scene=dict(xaxis=hidden, yaxis=hidden, zaxis=hidden, aspectmode='data'))
    fig.show()

def animate_views(vols, titles, interval=300):
    from matplotlib import animation
    frames = [_views(v, 'proj') for v in vols]
    lims = [(min(f[j].min() for f in frames), max(f[j].max() for f in frames)) for j in range(3)]
    fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.9))
    ims = []
    for j, ax in enumerate(axes):
        ims.append(ax.imshow(frames[0][j], cmap='gray', origin='lower', vmin=lims[j][0], vmax=lims[j][1]))
        ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f'projection along {"zyx"[j]}', fontsize=9)
    label = fig.suptitle(titles[0], fontsize=10)
    def update(i):
        for j in range(3):
            ims[j].set_data(frames[i][j])
        label.set_text(titles[i])
        return ims
    anim = animation.FuncAnimation(fig, update, frames=len(vols), interval=interval)
    plt.close(fig)
    display(HTML(anim.to_jshtml(default_mode='reflect')))

# ---- training --------------------------------------------------------------------------------
def checkpoints(run_dir):
    """epochs whose weights and latents are both on disk"""
    eps = []
    for f in glob.glob(f'{run_dir}/weights.*.pkl'):
        m = re.fullmatch(r'weights\.(\d+)\.pkl', os.path.basename(f))
        if m and os.path.exists(f'{run_dir}/z.{m.group(1)}.pkl'):
            eps.append(int(m.group(1)))
    return sorted(eps)

def training_pid(run_dir):
    """pid of the training job of this run if it is still alive"""
    for p in _TRAIN_PROCS.values():
        p.poll()  # reap a finished job, so that it does not linger as a zombie
    try:
        pid = int(open(f'{run_dir}/train.pid').read())
        with open(f'/proc/{pid}/cmdline', 'rb') as f:
            return pid if f'{run_dir}/train.sh'.encode() in f.read() else None
    except (OSError, ValueError):
        return None

def tail_text(path, nbytes=40000):
    if not os.path.exists(path):
        return ''
    with open(path, 'rb') as f:
        f.seek(max(0, os.path.getsize(path) - nbytes))
        return re.sub(r'\x1b\[[0-9;]*m', '', f.read().decode(errors='replace'))

def epoch_log(run_dir):
    """{(epoch, 'training'|'validation'): (gen_loss, snr, duration)} from run.log"""
    rows = {}
    if os.path.exists(f'{run_dir}/run.log'):
        pat = re.compile(r'Epoch: (\d+) Average (training|validation) gen_loss = ([-\d.e+]+), SNR2 = ([-\d.e+]+).*Finished in ([\d:]+)')
        for line in open(f'{run_dir}/run.log'):
            m = pat.search(line)
            if m:
                rows[(int(m.group(1)), m.group(2))] = (float(m.group(3)), float(m.group(4)), m.group(5))
    return rows

def report_training(name):
    run = S['runs'][name]
    rd = run['dir']
    pid = training_pid(rd)
    log = tail_text(f'{rd}/train_stdout.log')
    rows = epoch_log(rd)
    done = sorted(e for e, kind in rows if kind == 'training')
    finished = '__EXIT_CODE__' in log.rsplit('$ dsd train_multi', 1)[-1]
    state = 'TRAINING' if pid else ('finished' if finished and '__EXIT_CODE__ 0' in log[-200:] else 'stopped')
    print(f'Run "{name}": {state} · {len(done)} of {run["epochs"]} epochs logged · '
          f'checkpoints on disk: {checkpoints(rd) or "none yet"}')
    bars = [s.strip() for s in re.split(r'[\r\n]', log) if re.search(r'(Train|Validation) Epoch: \[', s)]
    if pid and bars:
        print('   ' + bars[-1][:170])
    if 'out of memory' in log.lower():
        print('\nThe GPU ran out of memory: lower BATCH_SIZE (or TEMPLATERES / DOWNFRAC) in 4.1 and start again.')
    if not pid and state == 'stopped' and log:
        print('\nLast lines of the training log:\n' + '\n'.join(re.split(r'[\r\n]+', log.strip())[-25:]))
    tmp = sorted(glob.glob(f'{rd}/tmp*.mrc'), key=os.path.getmtime)
    if not done and not tmp:
        return
    from matplotlib.ticker import MaxNLocator
    fig, axes = plt.subplots(1, 4 if tmp else 1, figsize=(14 if tmp else 4.5, 3), squeeze=False,
                             gridspec_kw=dict(width_ratios=[1.6, 1, 1, 1] if tmp else [1]))
    axes = axes[0]
    for kind, style in (('training', 'o-'), ('validation', 's--')):
        e = sorted(k for k, kk in rows if kk == kind)
        if e:
            axes[0].plot(e, [rows[(k, kind)][0] for k in e], style, ms=3, label=kind)
    axes[0].set_xlabel('epochs done'); axes[0].set_ylabel('reconstruction loss'); axes[0].legend(fontsize=8)
    axes[0].xaxis.set_major_locator(MaxNLocator(integer=True))
    if tmp:
        vol, _ = read_mrc(tmp[-1])
        for j, im in enumerate(_views(vol, 'proj')):
            axes[j + 1].imshow(im, cmap='gray', origin='lower'); axes[j + 1].set_xticks([]); axes[j + 1].set_yticks([])
        stamp = time.strftime('%H:%M', time.localtime(os.path.getmtime(tmp[-1])))
        axes[1].set_title(f'latest intermediate map ({stamp})', fontsize=9, loc='left')
    plt.tight_layout(); plt.show()

def watch_training(name, refresh=20):
    try:
        while True:
            pid = training_pid(S['runs'][name]['dir'])
            clear_output(wait=True)
            report_training(name)
            if not pid:
                break
            print(f'\n(refreshing every {refresh} s; stopping this cell does not stop training)')
            time.sleep(refresh)
    except KeyboardInterrupt:
        print('Stopped watching. Training goes on in the background: re-run 4.2 to watch, 4.3 to stop it.')

def status():
    d, p = S.get('data'), S.get('prep') or {}
    print(f'Project folder : {PROJECT_DIR}')
    if d:
        print(f'Data           : {d["name"]}: {d["n"]} particles, {d["D"]} px, {d["apix"]:.4g} Å/px')
        if d.get('local_stack') and not os.path.exists(d['local_stack']):
            print('                 the local copy of the particles is gone (new VM): re-run 2B')
    else:
        print('Data           : not chosen yet (step 2)')
    print('Poses and CTF  : ' + ('ready' if p.get('pose') else 'not yet (3.1)'))
    print('Mask           : ' + ('ready' if p.get('mask') else 'not yet (3.2)'))
    if p.get('mask_params'):
        print(f'Rigid bodies   : {p["num_bodies"]} (3.3)')
    for name, r in S['runs'].items():
        done = len({e for e, kind in epoch_log(r['dir']) if kind == 'training'})
        busy = 'TRAINING NOW · ' if training_pid(r['dir']) else ''
        print(f'Run {name:<11}: {busy}{done} of {r["epochs"]} epochs done · '
              f'checkpoints (by epoch index, from 0): {checkpoints(r["dir"]) or "-"}')
    if S.get('analysis'):
        a = S['analysis']
        print(f'Analysis       : run {a["run"]}, epoch {a["epoch"]}, {a["kmeans"]} clusters')

status()

## 2 · Data

Run **one** of the two cells below. **2A** makes a small synthetic dataset, so that you can check that everything works before spending GPU time on real data. **2B** uses your own particles.

OPUS-DSD needs particle images with poses and CTF parameters from a **consensus refinement without symmetry** (a 3D classification also works), as a RELION STAR file. Keep the consensus map (or a solvent mask made from it) at hand for step 3.2.

In [ ]:
#@title 2A · Demo: generate a synthetic dataset (≈1–2 min) { display-mode: "form" }
#@markdown A toy molecule with a **hinged arm** that swings by ±35° (continuous motion) and a **cap** domain that half of the particles lack (compositional change), projected in random orientations with a realistic CTF and noise. Written as a RELION 3.1 STAR file and `.mrcs` stack, together with the consensus map a refinement would give and a two-body definition for the optional multi-body step 3.3.
N_PARTICLES = 3000  #@param {type:"integer"}
SNR = 0.1  #@param {type:"number"}

import torch
from scipy.ndimage import binary_dilation
from cryodrgn import ctf as cctf

DEMO_D, DEMO_APIX, DEMO_SIGMA, ARM_RANGE, CAP_FRACTION = 128, 2.5, 7.0, 35.0, 0.5

def relion_matrix(rot, tilt, psi):
    """RELION's Euler_angles2matrix for arrays of angles in degrees"""
    a, b, g = (np.deg2rad(x) for x in (rot, tilt, psi))
    ca, sa, cb, sb, cg, sg = np.cos(a), np.sin(a), np.cos(b), np.sin(b), np.cos(g), np.sin(g)
    cc, cs, sc, ss = cb * ca, cb * sa, sb * ca, sb * sa
    A = np.empty((len(a), 3, 3))
    A[:, 0] = np.stack([cg * cc - sg * sa, cg * cs + sg * ca, -cg * sb], -1)
    A[:, 1] = np.stack([-sg * cc - cg * sa, -sg * cs + cg * ca, sg * sb], -1)
    A[:, 2] = np.stack([sc, ss, cb], -1)
    return A

def demo_parts(seed=0):
    """gaussian blobs (Å): a rigid core, an arm hinged to it and a cap; centred on the average structure"""
    rng = np.random.default_rng(seed)
    core = rng.normal(size=(28, 3)) * [22., 15., 12.]
    hinge = np.array([30., 0., 0.])
    arm = hinge + np.stack([np.linspace(8, 60, 8), np.zeros(8), np.linspace(0, 10, 8)], 1)
    cap = np.array([-22., 20., 14.]) + rng.normal(size=(7, 3)) * 5.
    centre = np.concatenate([core, arm, cap]).mean(0)
    return dict(core=core - centre, arm=arm - centre, hinge=hinge - centre, cap=cap - centre)

def arm_at(parts, theta):
    t = np.deg2rad(theta)
    Rz = np.array([[np.cos(t), -np.sin(t), 0], [np.sin(t), np.cos(t), 0], [0, 0, 1]])
    return (parts['arm'] - parts['hinge']) @ Rz.T + parts['hinge']

def render(pos, w, D=DEMO_D, apix=DEMO_APIX, sigma=DEMO_SIGMA):
    """sum of gaussians on a D^3 grid, computed separably as one matrix product"""
    c = (np.arange(D) - D // 2) * apix
    g = [np.exp(-(c[None] - pos[:, i, None]) ** 2 / (2 * sigma ** 2)) for i in range(3)]
    yx = (np.asarray(w)[:, None, None] * g[1][:, :, None] * g[0][:, None, :]).reshape(len(pos), -1)
    return (g[2].T @ yx).reshape(D, D, D).astype(np.float32)

def make_demo(out, n, snr, seed=0, D=DEMO_D, apix=DEMO_APIX, sigma=DEMO_SIGMA):
    os.makedirs(out, exist_ok=True)
    rng = np.random.default_rng(seed + 1)
    parts = demo_parts(seed)
    nc, na, ncap = len(parts['core']), len(parts['arm']), len(parts['cap'])
    theta = rng.uniform(-ARM_RANGE, ARM_RANGE, n)
    has_cap = rng.random(n) < CAP_FRACTION
    rot, psi = rng.uniform(-180, 180, n), rng.uniform(-180, 180, n)
    tilt = np.rad2deg(np.arccos(rng.uniform(-1, 1, n)))  # uniform viewing directions
    A = relion_matrix(rot, tilt, psi)
    dfu = rng.uniform(10000, 25000, n); dfv = dfu - rng.uniform(0, 300, n); dfang = rng.uniform(0, 180, n)
    volt, cs, w = 300., 2.7, 0.1

    c = torch.tensor((np.arange(D) - D // 2) * apix, dtype=torch.float32)
    yy, xx = torch.meshgrid(c, c, indexing='ij')
    k = torch.tensor((np.arange(D) - D // 2) / (D * apix), dtype=torch.float32)
    ky, kx = torch.meshgrid(k, k, indexing='ij')
    freqs = torch.stack([kx, ky], -1).view(-1, 2)
    f32 = lambda x: torch.tensor(x, dtype=torch.float32)
    shift = lambda x: torch.fft.fftshift(x, dim=(-2, -1))
    ishift = lambda x: torch.fft.ifftshift(x, dim=(-2, -1))
    imgs = np.empty((n, D, D), np.float32)
    for s in range(0, n, 64):
        e = min(s + 64, n)
        pos = np.stack([np.concatenate([parts['core'], arm_at(parts, theta[i]), parts['cap']]) for i in range(s, e)])
        wts = np.concatenate([np.ones((e - s, nc + na)), np.repeat(has_cap[s:e, None], ncap, 1)], 1)
        r = f32(np.einsum('nij,nmj->nmi', A[s:e], pos))  # a blob at p lands at A p in the image frame
        d2 = (xx - r[..., 0, None, None]) ** 2 + (yy - r[..., 1, None, None]) ** 2
        proj = (f32(wts)[..., None, None] * torch.exp(-d2 / (2 * sigma ** 2))).sum(1)
        ctf = cctf.compute_ctf(freqs[None], f32(dfu[s:e, None]), f32(dfv[s:e, None]), f32(dfang[s:e, None]),
                               f32(volt), f32(cs), f32(w)).view(-1, D, D)
        imgs[s:e] = shift(torch.fft.ifft2(ishift(shift(torch.fft.fft2(ishift(proj))) * ctf))).real.numpy()
    imgs += rng.normal(scale=np.sqrt(imgs.var() / snr), size=imgs.shape).astype(np.float32)
    cmrc.write(f'{out}/particles.mrcs', imgs, Apix=apix, is_vol=False)

    with open(f'{out}/particles.star', 'w') as f:
        f.write('\n# version 30001\n\ndata_optics\n\nloop_\n')
        optics = ['_rlnOpticsGroupName', '_rlnOpticsGroup', '_rlnMicrographOriginalPixelSize', '_rlnVoltage',
                  '_rlnSphericalAberration', '_rlnAmplitudeContrast', '_rlnImagePixelSize', '_rlnImageSize',
                  '_rlnImageDimensionality']
        f.write(''.join(f'{h} #{i}\n' for i, h in enumerate(optics, 1)))
        f.write(f'opticsGroup1 1 {apix:.4f} {volt:.1f} {cs:.2f} {w:.2f} {apix:.4f} {D} 2\n\n\n')
        f.write('# version 30001\n\ndata_particles\n\nloop_\n')
        cols = ['_rlnImageName', '_rlnAngleRot', '_rlnAngleTilt', '_rlnAnglePsi', '_rlnOriginXAngst',
                '_rlnOriginYAngst', '_rlnDefocusU', '_rlnDefocusV', '_rlnDefocusAngle', '_rlnPhaseShift',
                '_rlnOpticsGroup', '_rlnRandomSubset']
        f.write(''.join(f'{h} #{i}\n' for i, h in enumerate(cols, 1)))
        for i in range(n):
            f.write(f'{i + 1:06d}@particles.mrcs {rot[i]:.6f} {tilt[i]:.6f} {psi[i]:.6f} 0.000000 0.000000 '
                    f'{dfu[i]:.2f} {dfv[i]:.2f} {dfang[i]:.2f} 0.000000 1 {i % 2 + 1}\n')
        f.write('\n')

    # what a consensus refinement would give: the arm smeared over its range, the cap at half occupancy
    thetas = np.linspace(-ARM_RANGE, ARM_RANGE, 15)
    arms = np.concatenate([arm_at(parts, t) for t in thetas])
    core_map = render(np.concatenate([parts['core'], parts['cap']]), np.r_[np.ones(nc), np.full(ncap, CAP_FRACTION)])
    arm_map = render(arms, np.full(len(arms), 1. / len(thetas)))
    write_mrc(f'{out}/consensus.mrc', core_map + arm_map, apix)
    # two rigid bodies for multi-body refinement: core (with the cap) and the arm, like RELION's body masks
    core_body = binary_dilation(core_map > 0.15, iterations=2)
    arm_body = binary_dilation(arm_map > 0.05, iterations=2) & ~(core_map > arm_map)
    write_mrc(f'{out}/body1_core.mrc', core_body & ~arm_body, apix)
    write_mrc(f'{out}/body2_arm.mrc', arm_body, apix)
    with open(f'{out}/bodies.star', 'w') as f:
        f.write('\ndata_\n\nloop_\n_rlnBodyMaskName #1\n_rlnBodyRotateRelativeTo #2\n_rlnBodySigmaAngles #3\n'
                '_rlnBodySigmaOffset #4\nbody1_core.mrc 2 10 2\nbody2_arm.mrc 1 10 2\n\n')
    np.savez(f'{out}/ground_truth.npz', arm_angle=theta, has_cap=has_cap)
    return imgs

demo_dir = f'{PROJECT_DIR}/demo_data'
params = dict(n=N_PARTICLES, snr=SNR, version=1)
if os.path.exists(f'{demo_dir}/params.json') and json.load(open(f'{demo_dir}/params.json')) == params:
    print('The demo dataset already exists, reusing it.')
else:
    t0 = time.time()
    make_demo(demo_dir, N_PARTICLES, SNR)
    json.dump(params, open(f'{demo_dir}/params.json', 'w'))
    print(f'Generated {N_PARTICLES} particles in {time.time() - t0:.0f} s.')

sig = f'demo:{N_PARTICLES}:{SNR}'
if (S.get('data') or {}).get('sig') != sig:
    S['prep'], S['analysis'] = {}, {}
S['data'] = dict(kind='demo', sig=sig, name='synthetic demo', dir=demo_dir, star=f'{demo_dir}/particles.star',
                 source_star=f'{demo_dir}/particles.star', particles=f'{demo_dir}/particles.mrcs', relion31=True,
                 D=DEMO_D, apix=DEMO_APIX, n=N_PARTICLES, indices=None, local_stack=None, phase_shift=True,
                 consensus=f'{demo_dir}/consensus.mrc', bodies_star=f'{demo_dir}/bodies.star')
save_session()

stack = cmrc.parse_mrc(S['data']['particles'], lazy=True)[0]
show_gallery([stack[i].get() for i in range(8)], [f'particle {i}' for i in range(8)], size=1.4)
show_views([read_mrc(S['data']['consensus'])[0]], ['consensus map'])

In [ ]:
#@title 2B · Your data: a RELION particle STAR file { display-mode: "form" }
#@markdown Path of the STAR file of your consensus refinement, e.g. on Drive `/content/drive/MyDrive/myproject/run_data.star` (find it in the file browser on the left, right-click › *Copy path*).
STAR_FILE = ""  #@param {type:"string"}
#@markdown Folder the image paths in the STAR file are relative to, usually your RELION project folder (blank: look in the STAR file's folder and the three above it).
PARTICLES_ROOT = ""  #@param {type:"string"}
#@markdown Pixel size in Å (0: read it from the STAR file or the image header).
PIXEL_SIZE = 0  #@param {type:"number"}
#@markdown Use a random subset of this many particles for a quick trial (0: all of them).
MAX_PARTICLES = 0  #@param {type:"integer"}
#@markdown Gather the particles into one stack on the VM's local disk. Much faster to train from than Drive, but it has to be redone (re-run this cell) after a disconnect.
COPY_TO_LOCAL_DISK = True  #@param {type:"boolean"}

def _load_user_data():
    if not STAR_FILE or not os.path.isfile(STAR_FILE):
        raise FileNotFoundError(f'STAR_FILE not found: "{STAR_FILE}"')
    with open(STAR_FILE) as f:
        relion31 = any(line.startswith('data_optics') for line in f)
    try:
        s = cstar.Starfile.load(STAR_FILE, relion31=relion31)
    except Exception as e:
        raise RuntimeError(f'Could not parse {STAR_FILE} ({e}). OPUS-DSD reads RELION 3.0/3.1 style particle STAR '
                           'files with one loop of particles.') from e
    hdrs = set(s.headers)
    optics_hdrs = set(s.optics_header) if relion31 else hdrs
    missing = [c for c in ('_rlnImageName', '_rlnAngleRot', '_rlnAngleTilt', '_rlnAnglePsi', '_rlnDefocusU',
                           '_rlnDefocusV', '_rlnDefocusAngle') if c not in hdrs]
    missing += [c for c in ('_rlnVoltage', '_rlnSphericalAberration', '_rlnAmplitudeContrast') if c not in optics_hdrs]
    if missing:
        raise RuntimeError(f'The STAR file lacks {missing}. It must come from a refinement (poses) with CTF parameters.')
    if relion31 and len(s.optics_df) > 1:
        keys = [c for c in ('_rlnVoltage', '_rlnSphericalAberration', '_rlnAmplitudeContrast',
                            '_rlnImagePixelSize', '_rlnImageSize') if c in s.optics_df]
        if any(s.optics_df[c].astype(float).nunique() > 1 for c in keys):
            raise RuntimeError('The optics groups differ in voltage, Cs, amplitude contrast, pixel or box size; '
                               'OPUS-DSD reads a single optics group.')
        print(f'{len(s.optics_df)} optics groups with the same optics: treating them as one.')
        s.optics_df = s.optics_df.iloc[:1].reset_index(drop=True)
        if '_rlnOpticsGroup' in s.df:
            s.df['_rlnOpticsGroup'] = '1'

    names = s.df['_rlnImageName'].astype(str)
    img_idx = names.str.split('@').str[0].astype(int).to_numpy() - 1
    img_file = names.str.split('@').str[1].to_numpy()
    star_dir = os.path.dirname(os.path.abspath(STAR_FILE))
    roots = [PARTICLES_ROOT] if PARTICLES_ROOT else [os.path.normpath(os.path.join(star_dir, *['..'] * k)) for k in range(4)]
    f0 = img_file[0]
    root = next((r for r in roots if any(os.path.isfile(c) for c in
                 ([f0] if os.path.isabs(f0) else []) + [os.path.join(r, f0), os.path.join(r, os.path.basename(f0))])), roots[0])
    if root != star_dir:
        print(f'Image paths are relative to {root}')
    resolved = {}
    for f in dict.fromkeys(img_file):
        cands = ([f] if os.path.isabs(f) else []) + [os.path.join(root, f), os.path.join(root, os.path.basename(f))]
        resolved[f] = next((c for c in cands if os.path.isfile(c)), None)
    lost = [f for f, r in resolved.items() if r is None]
    if lost:
        raise FileNotFoundError(f'{len(lost)} of {len(resolved)} particle stacks were not found, e.g. {lost[:3]} '
                                f'(looked in {root}). Set PARTICLES_ROOT to the folder the paths are relative to.')
    h = cmrc.parse_header(resolved[img_file[0]])
    D = int(h.fields['nx'])
    if D % 2:
        raise RuntimeError(f'The particle box ({D} px) must be even.')
    apix = PIXEL_SIZE
    if not apix and relion31 and '_rlnImagePixelSize' in s.optics_header:
        apix = float(s.optics_df['_rlnImagePixelSize'][0])
    if not apix and {'_rlnDetectorPixelSize', '_rlnMagnification'} <= hdrs:
        apix = float(s.df['_rlnDetectorPixelSize'][0]) * 1e4 / float(s.df['_rlnMagnification'][0])
    if not apix:
        apix = float(h.get_apix())
    if not apix or apix == 1.0:
        print(f'WARNING: pixel size {apix} Å may be a placeholder; set PIXEL_SIZE if it is wrong.')

    N = len(s.df)
    sel = np.arange(N)
    if 0 < MAX_PARTICLES < N:
        sel = np.sort(np.random.default_rng(0).choice(N, MAX_PARTICLES, replace=False))
    data_dir = f'{PROJECT_DIR}/data'
    os.makedirs(data_dir, exist_ok=True)
    np.save(f'{data_dir}/source_indices.npy', sel)
    work = cstar.Starfile(s.headers, s.df.iloc[sel].reset_index(drop=True),
                          optics=(s.optics_header, s.optics_df) if relion31 else None, relion31=relion31)
    sig = f'user:{os.path.abspath(STAR_FILE)}:{os.path.getmtime(STAR_FILE):.0f}:{len(sel)}:{root}'

    local_stack = None
    if COPY_TO_LOCAL_DISK:
        local_stack = f'{LOCAL_DIR}/particles.mrcs'
        if os.path.exists(f'{local_stack}.json') and json.load(open(f'{local_stack}.json')).get('sig') == sig:
            print('The local particle stack is up to date.')
        else:
            size = len(sel) * D * D * 4
            free = shutil.disk_usage(LOCAL_DIR).free
            if size > 0.9 * free:
                raise RuntimeError(f'The stack needs {size / 2**30:.1f} GB but the VM disk has {free / 2**30:.1f} GB free: '
                                   'use MAX_PARTICLES, or untick COPY_TO_LOCAL_DISK.')
            from tqdm.auto import tqdm
            header = cmrc.MRCHeader.make_default_header(types.SimpleNamespace(shape=(len(sel), D, D)),
                                                        is_vol=False, Apix=apix)
            layout, handles = {}, {}
            with open(f'{local_stack}.part', 'wb') as out:
                header.write(out)
                for i in tqdm(sel, desc='copying particles', mininterval=2):
                    path = resolved[img_file[i]]
                    if path not in layout:
                        hh = cmrc.parse_header(path)
                        layout[path] = (np.dtype(hh.dtype), 1024 + hh.fields['next'])
                    fh = handles.get(path)
                    if fh is None:
                        if len(handles) >= 64:
                            handles.pop(next(iter(handles))).close()
                        fh = handles[path] = open(path, 'rb')
                    dt, start = layout[path]
                    fh.seek(start + int(img_idx[i]) * D * D * dt.itemsize)
                    out.write(np.fromfile(fh, dtype=dt, count=D * D).astype(np.float32).tobytes())
            for fh in handles.values():
                fh.close()
            os.replace(f'{local_stack}.part', local_stack)
            json.dump(dict(sig=sig), open(f'{local_stack}.json', 'w'))
        work.df['_rlnImageName'] = [f'{k + 1:06d}@{local_stack}' for k in range(len(sel))]
        particles = local_stack
    else:
        paths = [resolved[img_file[i]] for i in sel]
        if any(re.search(r'\s', p) for p in paths):
            raise RuntimeError('Some particle paths contain spaces, which STAR files cannot hold: tick COPY_TO_LOCAL_DISK.')
        work.df['_rlnImageName'] = [f'{img_idx[i] + 1:06d}@{p}' for i, p in zip(sel, paths)]
        particles = f'{data_dir}/particles.star'
    work.write(f'{data_dir}/particles.star')

    if (S.get('data') or {}).get('sig') != sig:
        S['prep'], S['analysis'] = {}, {}
    S['data'] = dict(kind='user', sig=sig, name=os.path.basename(STAR_FILE), dir=data_dir,
                     star=f'{data_dir}/particles.star', source_star=os.path.abspath(STAR_FILE),
                     source_relion31=relion31, particles=particles, relion31=relion31, D=D, apix=apix, n=len(sel),
                     indices=f'{data_dir}/source_indices.npy', local_stack=local_stack,
                     phase_shift='_rlnPhaseShift' in hdrs, consensus=None, bodies_star=None)
    save_session()
    print(f'{len(sel)} of {N} particles · box {D} px · {apix:.4g} Å/px · RELION {"3.1+" if relion31 else "3.0"} STAR')

_load_user_data()
from cryodrgn import dataset as cdataset
_first = cdataset.load_particles(S['data']['star'], lazy=True, relion31=S['data']['relion31'])[:8]
show_gallery([x.get() for x in _first], [f'particle {i}' for i in range(len(_first))], size=1.4)

## 3 · Prepare the inputs

In [ ]:
#@title 3.1 · Convert the STAR file into pose and CTF pickles { display-mode: "form" }
#@markdown Runs `dsd parse_pose_star` and `dsd parse_ctf_star` (what `dsdsh prepare` does under the hood) and plots the viewing directions, defoci and shifts as a sanity check.
def _prepare():
    d = need('data', '2A or 2B')
    prep = f'{PROJECT_DIR}/prep'
    os.makedirs(prep, exist_ok=True)
    r31 = '--relion31' if d['relion31'] else ''
    common = f'{q(d["star"])} -D {d["D"]} --Apix {d["apix"]} {r31}'
    sh(f'dsd parse_pose_star {common} -o {prep}/pose_euler.pkl', quiet=True)
    sh(f'dsd parse_ctf_star {common} -o {prep}/ctf.pkl {"" if d["phase_shift"] else "--ps 0"}', quiet=True)
    rot, trans, euler = pickle.load(open(f'{prep}/pose_euler.pkl', 'rb'))[:3]
    ctf = pickle.load(open(f'{prep}/ctf.pkl', 'rb'))
    assert len(euler) == len(ctf) == d['n'], 'the number of poses or CTFs does not match the number of particles'
    S['prep'] = {**(S.get('prep') or {}), 'pose': f'{prep}/pose_euler.pkl', 'ctf': f'{prep}/ctf.pkl'}
    save_session()
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
    ax[0].hexbin(euler[:, 0], euler[:, 1], gridsize=36, bins='log', cmap='viridis')
    ax[0].set_xlabel('rot (°)'); ax[0].set_ylabel('tilt (°)'); ax[0].set_title('viewing directions', fontsize=10)
    ax[1].hist((ctf[:, 2] + ctf[:, 3]) / 2e4, bins=50, color='tab:blue')
    ax[1].set_xlabel('defocus (µm)'); ax[1].set_title('defocus', fontsize=10)
    ax[2].hist(np.hypot(trans[:, 0], trans[:, 1]) * d['D'], bins=50, color='tab:orange')
    ax[2].set_xlabel('shift (px)'); ax[2].set_title('in-plane shifts', fontsize=10)
    plt.tight_layout(); plt.show()
    print(f'Voltage {ctf[0, 5]:.0f} kV · Cs {ctf[0, 6]:.2f} mm · amplitude contrast {ctf[0, 7]:.2f} · '
          f'pixel size {ctf[0, 1]:.4g} Å')

_prepare()

In [ ]:
#@title 3.2 · Solvent mask { display-mode: "form" }
#@markdown OPUS-DSD reconstructs what lies inside this mask, and crops the images to its extent, so it must enclose every conformation. Give a mask you already have, **or** your consensus map to build one from (the demo provides its own map).
MASK_FILE = ""  #@param {type:"string"}
CONSENSUS_MAP = ""  #@param {type:"string"}
#@markdown Used when building the mask: THRESHOLD 0 picks one automatically, otherwise give a contour level as you would in ChimeraX (of the low-passed map).
THRESHOLD = 0  #@param {type:"number"}
LOWPASS_A = 15  #@param {type:"number"}
DILATE_A = 6  #@param {type:"number"}
SOFT_EDGE_A = 8  #@param {type:"number"}

def _mask():
    from scipy.ndimage import gaussian_filter
    d = need('data', '2A or 2B')
    need('prep', '3.1')
    D, apix = d['D'], d['apix']
    prep = f'{PROJECT_DIR}/prep'
    ref = None
    src = CONSENSUS_MAP or d.get('consensus')
    if src:
        ref, ref_apix = read_mrc(src)
        if ref.shape != (D, D, D) or abs(ref_apix - apix) > 1e-3 * apix:
            print(f'Resampling the map from {ref.shape[0]} px at {ref_apix:.4g} Å to {D} px at {apix:.4g} Å.')
            ref = fit_to_box(ref, ref_apix, D, apix)
    if MASK_FILE:
        m, m_apix = read_mrc(MASK_FILE)
        if m.shape != (D, D, D) or abs(m_apix - apix) > 1e-3 * apix:
            print(f'Resampling the mask from {m.shape[0]} px at {m_apix:.4g} Å to {D} px at {apix:.4g} Å.')
            m = fit_to_box(m, m_apix, D, apix)
        write_mrc(f'{prep}/mask.mrc', np.clip(m, 0, 1), apix)
    elif ref is not None:
        lp = gaussian_filter(ref, 0.187 * LOWPASS_A / apix) if LOWPASS_A > 0 else ref
        thr = THRESHOLD or 0.5 * iso_level(lp)
        write_mrc(f'{prep}/map_lowpass.mrc', lp, apix)
        sh(f'dsdsh create_mask {prep}/map_lowpass.mrc -o {prep}/mask.mrc --threshold {thr:.6g} --largest-component '
           f'--fill-holes --dilate {max(0, round(DILATE_A / apix))} --soft-edge {SOFT_EDGE_A / apix:.2f}', quiet=True)
        print(f'Threshold {thr:.4g} on the map low-passed to {LOWPASS_A} Å.')
    else:
        raise RuntimeError('Set MASK_FILE, or CONSENSUS_MAP to build a mask from.')
    m, _ = read_mrc(f'{prep}/mask.mrc')
    nz = np.argwhere(m > 0)
    if not len(nz):
        raise RuntimeError('The mask is empty: lower THRESHOLD.')
    margin = min(nz.min(), D - 1 - nz.max())
    print(f'Mask: {100 * (m > 0.5).mean():.2f}% of the box, extends to {margin} px from the box edge.')
    if margin < 2:
        print('WARNING: the mask touches the edge of the box. Lower DILATE_A / SOFT_EDGE_A or raise THRESHOLD.')
    S['prep']['mask'] = f'{prep}/mask.mrc'
    save_session()
    show_views([ref if ref is not None else m], ['map with mask' if ref is not None else 'mask'], 'slice', masks=[[m]])
    if ref is not None:
        show_views([ref], ['projections'], 'proj', masks=[[m]])

_mask()

In [ ]:
#@title 3.3 · *Optional* · Rigid bodies for multi-body dynamics (OPUS-DSD2) { display-mode: "form" }
#@markdown Skip this unless you want OPUS-DSD2 to model how rigid bodies move. It takes the body definition STAR file of a RELION multi-body refinement (columns `_rlnBodyMaskName` and `_rlnBodyRotateRelativeTo`, mask paths relative to that file; only the body masks are needed, not the refinement itself). The demo comes with one (core and arm), used when BODY_STAR is blank.
BODY_STAR = ""  #@param {type:"string"}
#@markdown Body the others move relative to, numbered from 1 (0: the one referenced most often).
ORIGIN_BODY = 0  #@param {type:"integer"}
#@markdown Optional: folder of PC-traversal maps (e.g. `.../analyze.N/pc1` of an earlier run) whose motion sets the rotation axes.
TRAVERSAL_MAPS = ""  #@param {type:"string"}

def _bodies():
    d = need('data', '2A or 2B')
    p = need('prep', '3.1')
    body_star = BODY_STAR or d.get('bodies_star')
    if not body_star:
        raise RuntimeError('Set BODY_STAR to the body definition STAR file.')
    b = cstar.Starfile.load(body_star)
    for c in ('_rlnBodyMaskName', '_rlnBodyRotateRelativeTo'):
        if c not in b.headers:
            raise RuntimeError(f'{body_star} has no {c} column')
    nb = len(b.df)
    masks = [os.path.join(os.path.dirname(body_star), x) for x in b.df['_rlnBodyMaskName']]
    for m in masks:
        if not os.path.isfile(m):
            raise FileNotFoundError(m)
    prep = f'{PROJECT_DIR}/prep'
    origin = ORIGIN_BODY or (1 if d['kind'] == 'demo' else 0)
    cmd = (f'python -m cryodrgn.commands.parse_multi_pose_star {q(d["star"])} -D {d["D"]} --Apix {d["apix"]} '
           f'-o {prep}/pose_euler_multibody.pkl --masks {q(body_star)} --bodies {nb} --outmasks mask_params '
           + ('--relion31 ' if d['relion31'] else '') + (f'--origin-body {origin} ' if origin else '')
           + (f'--volumes {q(TRAVERSAL_MAPS)}' if TRAVERSAL_MAPS else ''))
    sh(cmd, quiet=True)
    shutil.copy(os.path.join(os.path.dirname(body_star), 'mask_params.pkl'), f'{prep}/mask_params.pkl')
    p.update(pose_multibody=f'{prep}/pose_euler_multibody.pkl', mask_params=f'{prep}/mask_params.pkl', num_bodies=nb)
    save_session()
    print(f'{nb} rigid bodies defined. Tick MULTI_BODY in 4.1 to train with them.')
    body_vols = [read_mrc(m)[0] for m in masks]
    ref = read_mrc(d['consensus'])[0] if d.get('consensus') else sum(body_vols)
    show_views([ref], [f'{nb} bodies'], 'proj', masks=[body_vols])

_bodies()

## 4 · Train

Training runs as a background job: closing the browser tab or stopping a cell does not stop it, but the VM going away does. Checkpoints (`weights.N.pkl`, `z.N.pkl`, numbered from epoch 0) go to `runs/RUN_NAME` in the project folder after every epoch, and 4.1 continues from the newest one when it is re-run with **RESUME** ticked.

The main settings, from the [OPUS-DSD README](https://github.com/alncat/opusDSD#training):

| Setting | Meaning |
|---|---|
| `BATCH_SIZE` | images per batch; the first thing to lower if the GPU runs out of memory |
| `ZDIM` | dimension of the latent space |
| `BETA_CONTROL` | strength of the β-VAE prior, roughly 0.5–4; higher for data with higher SNR |
| `LAMBDA` | strength of the structural-disentanglement prior, roughly 0.1–3; lower it for noisy data, or if the UMAP comes out as a thin ribbon |
| `BFACTOR` | B-factor applied to the reconstruction, roughly 3–6; higher values sharpen but reveal weak density later |
| `DOWNFRAC` | images are downsampled to `box × DOWNFRAC` pixels, which must be at least 128 (0: automatic, at most 192) |
| `TEMPLATERES` | size of the decoder's output volume, a multiple of 16 from 128 to 256 (0: automatic, from the mask) |

Each checkpoint takes about 0.8 GB, so only the newest `KEEP_CHECKPOINTS` are kept (0 keeps all). Files deleted on Drive go to its trash: empty it if you run short of space.

In [ ]:
#@title 4.1 · Train (runs in the background) { display-mode: "form" }
RUN_NAME = "run1"  #@param {type:"string"}
#@markdown Total number of passes over the particles; the README's examples use 20.
EPOCHS = 12  #@param {type:"integer"}
BATCH_SIZE = 8  #@param {type:"integer"}
ZDIM = 12  #@param {type:"integer"}
LEARNING_RATE = 0.0001  #@param {type:"number"}
BETA_CONTROL = 2.0  #@param {type:"number"}
LAMBDA = 1.0  #@param {type:"number"}
BFACTOR = 4.0  #@param {type:"number"}
VALIDATION_FRACTION = 0.2  #@param {type:"number"}
DOWNFRAC = 0  #@param {type:"number"}
TEMPLATERES = 0  #@param {type:"integer"}
#@markdown OPUS-DSD2 multi-body dynamics, with the bodies of step 3.3 and a dynamics latent space of ZAFFDIM dimensions.
MULTI_BODY = False  #@param {type:"boolean"}
ZAFFDIM = 6  #@param {type:"integer"}
KEEP_CHECKPOINTS = 2  #@param {type:"integer"}
#@markdown Continue RUN_NAME from its newest checkpoint, if it has one.
RESUME = True  #@param {type:"boolean"}
#@markdown Any other `dsd train_multi` options, e.g. `--accum-step 4`.
EXTRA_ARGS = ""  #@param {type:"string"}

def _train():
    import torch, psutil
    d = need('data', '2A or 2B')
    p = need('prep', '3.1')
    if not p.get('mask'):
        raise RuntimeError('Make a mask first (3.2).')
    if not torch.cuda.is_available():
        raise RuntimeError('Training needs a GPU: Runtime > Change runtime type > GPU. The new runtime starts '
                           'empty, so run step 1 again (and 2B if you copied particles to the local disk).')
    if d.get('local_stack') and not os.path.exists(d['local_stack']):
        raise RuntimeError('The local copy of the particles is gone (new VM): re-run 2B first.')
    if not re.fullmatch(r'[\w.-]+', RUN_NAME):
        raise ValueError('RUN_NAME may only contain letters, digits, "_", "-" and "."')
    rd = f'{PROJECT_DIR}/runs/{RUN_NAME}'
    os.makedirs(rd, exist_ok=True)
    S['current_run'] = RUN_NAME
    if training_pid(rd):
        print(f'Run "{RUN_NAME}" is already training; watching it.')
        return True

    D = d['D']
    arch = dict(zdim=ZDIM, multibody=MULTI_BODY, zaffdim=ZAFFDIM)
    arch['downfrac'] = DOWNFRAC or float(np.ceil(max(128 / D, min(0.75, 192 / D)) * 1e4) / 1e4)
    if arch['downfrac'] * D < 128:
        raise ValueError(f'DOWNFRAC × box must be at least 128 px, i.e. DOWNFRAC ≥ {128 / D:.4f}')
    # the decoder output size, as the model would guess it from the extent of the mask
    m, _ = read_mrc(p['mask'])
    nz = np.argwhere(m != 0)
    margin = min(nz.min(), (D - nz.max(0)).min())
    window = min((D - 2 * margin + 4) / D, 0.9)
    render = int(D * arch['downfrac']) // 2 * 2
    guess = int(np.ceil((int(render * window) // 2 * 2) / 0.8 / 16) * 16)
    arch['templateres'] = TEMPLATERES or int(min(256, max(128, guess)))
    if arch['templateres'] % 16:
        raise ValueError('TEMPLATERES must be a multiple of 16')
    if MULTI_BODY and not p.get('mask_params'):
        raise RuntimeError('MULTI_BODY needs the rigid bodies of step 3.3.')

    ck = checkpoints(rd)
    run = S['runs'].get(RUN_NAME)
    resume = ''
    if ck and not RESUME:
        raise RuntimeError(f'Run "{RUN_NAME}" already has checkpoints (epochs {ck}). Tick RESUME to continue it, '
                           'or choose a new RUN_NAME.')
    if ck:
        if not run or run.get('data_sig') != d['sig']:
            raise RuntimeError(f'Run "{RUN_NAME}" was trained on other data; choose a new RUN_NAME.')
        if run['arch'] != arch:
            print(f'Resuming with the architecture the run started with: {run["arch"]}')
            arch = run['arch']
        if ck[-1] + 1 >= EPOCHS:
            print(f'Run "{RUN_NAME}" has already trained {ck[-1] + 1} epochs; raise EPOCHS to train more.')
            return True
        resume = f'--load {rd}/weights.{ck[-1]}.pkl --latents {rd}/z.{ck[-1]}.pkl'
        print(f'Resuming after epoch {ck[-1]}.')

    particles = d['particles']
    poses = p['pose_multibody'] if arch['multibody'] else p['pose']
    inmem = d['n'] * D * D * 4 < 0.35 * psutil.virtual_memory().available
    cmd = (f'dsd train_multi {q(particles)} --ctf {p["ctf"]} --poses {poses} -r {p["mask"]} -o {rd} '
           f'--split {rd}/split.pkl -n {EPOCHS} -b {BATCH_SIZE} --zdim {arch["zdim"]} --lr {LEARNING_RATE} '
           f'--num-gpus {max(1, torch.cuda.device_count())} --multigpu --beta-control {BETA_CONTROL} --beta cos '
           f'--lamb {LAMBDA} --bfactor {BFACTOR} --downfrac {arch["downfrac"]} --valfrac {VALIDATION_FRACTION} '
           f'--templateres {arch["templateres"]} --tmp-prefix {rd}/tmp')
    if particles.endswith('.star'):
        cmd += ' --relion31' * d['relion31']
    if arch['multibody']:
        cmd += f' --masks {p["mask_params"]} --zaffdim {arch["zaffdim"]}'
    cmd += ' --inmem' * inmem + f' {resume} {EXTRA_ARGS}'.rstrip()

    prune = (f'ls -1 weights.*.pkl 2>/dev/null | sort -t. -k2 -n | head -n -{KEEP_CHECKPOINTS} | xargs -r rm -f'
             if KEEP_CHECKPOINTS > 0 else ':')
    with open(f'{rd}/train.sh', 'w') as f:
        f.write(f'#!/bin/bash\ncd {rd}\necho "=== $(date)" >> train_stdout.log\n'
                f'cat train_cmd.txt >> train_stdout.log\n{cmd} >> train_stdout.log 2>&1 &\nJOB=$!\n'
                f'while kill -0 $JOB 2>/dev/null; do sleep 30; {prune}; done\n'
                f'wait $JOB; CODE=$?\n{prune}\necho "__EXIT_CODE__ $CODE" >> train_stdout.log\n')
    with open(f'{rd}/train_cmd.txt', 'w') as f:
        f.write(f'$ {cmd}\n')
    proc = subprocess.Popen(['bash', f'{rd}/train.sh'], start_new_session=True, stdout=subprocess.DEVNULL,
                            stderr=subprocess.DEVNULL, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    _TRAIN_PROCS[RUN_NAME] = proc
    with open(f'{rd}/train.pid', 'w') as f:
        f.write(str(proc.pid))
    S['runs'][RUN_NAME] = dict(dir=rd, epochs=EPOCHS, data_sig=d['sig'], arch=arch, started=time.ctime())
    save_session()
    print(f'$ {cmd}\n')
    render = int(D * arch['downfrac']) // 2 * 2
    print(f'Images are downsampled to {render} px ({d["apix"] / arch["downfrac"]:.3g} Å/px); the decoder outputs '
          f'{arch["templateres"]}³ voxels; particles are read {"into memory" if inmem else "from disk"}.')
    time.sleep(5)
    return True

if _train():
    watch_training(S['current_run'])

In [ ]:
#@title 4.2 · Watch training (safe to stop and re-run) { display-mode: "form" }
REFRESH_SECONDS = 30  #@param {type:"integer"}
watch_training(need('current_run', '4.1'), REFRESH_SECONDS)

In [ ]:
#@title 4.3 · Stop training { display-mode: "form" }
def _stop():
    name = need('current_run', '4.1')
    pid = training_pid(S['runs'][name]['dir'])
    if not pid:
        print(f'Run "{name}" is not training.')
        return
    os.killpg(os.getpgid(pid), signal.SIGTERM)
    time.sleep(3)
    print(f'Stopped run "{name}". Re-run 4.1 with RESUME ticked to continue from the newest checkpoint.')

_stop()

## 5 · Analyze the latent space

Runs `dsdsh analyze`: PCA and UMAP of the latent codes, k-means clustering, and the latent points along each principal component that step 6 turns into maps. Training does not have to be finished, any saved epoch can be analyzed.

For the demo, two extra panels colour the UMAP by the true arm angle and by whether the cap is present. Once the model has learned the heterogeneity, particles with and without the cap separate and the arm angle changes smoothly across the map. Each particle's latent code is a running average that starts from random noise, so after only a few epochs both panels still look mixed.

In [ ]:
#@title 5.1 · Analyze { display-mode: "form" }
#@markdown Epoch to analyze (-1: the newest checkpoint), number of principal components to traverse, number of k-means clusters, and points per PC traversal.
EPOCH = -1  #@param {type:"integer"}
NUM_PCS = 4  #@param {type:"integer"}
NUM_CLUSTERS = 16  #@param {type:"integer"}
SAMPLES_PER_PC = 10  #@param {type:"integer"}
#@markdown Reuse the UMAP of an earlier analysis of the same epoch (it is the slow part for large datasets).
REUSE_UMAP = False  #@param {type:"boolean"}

def _analyze():
    name = need('current_run', '4.1')
    rd = S['runs'][name]['dir']
    ck = checkpoints(rd)
    if not ck:
        raise RuntimeError('No checkpoint yet: train for at least one epoch (step 4).')
    N = ck[-1] if EPOCH < 0 else EPOCH
    if N not in ck:
        raise RuntimeError(f'No checkpoint for epoch {N}; available: {ck}')
    adir = f'{rd}/analyze.{N}'
    skip = '--skip-umap' if REUSE_UMAP and os.path.exists(f'{adir}/umap.pkl') else ''
    sh(f'dsdsh analyze {rd} {N} {NUM_PCS} {NUM_CLUSTERS} --psample {SAMPLES_PER_PC} {skip}', quiet=True)
    kdir = f'{adir}/kmeans{NUM_CLUSTERS}'
    for f in (f'{adir}/umap.pkl', f'{kdir}/labels.pkl', f'{kdir}/centers.txt'):
        if not os.path.exists(f):
            raise RuntimeError(f'dsdsh analyze did not write {f}')
    S['analysis'] = dict(run=name, epoch=N, dir=adir, kmeans=NUM_CLUSTERS, pcs=NUM_PCS)
    save_session()
    print(f'Results in {adir}')
    show_pngs([f'{rd}/train_losses.png', f'{adir}/z_pca.png', f'{adir}/umap.png', f'{kdir}/umap.png'], width=360)

    labels = pickle.load(open(f'{kdir}/labels.pkl', 'rb'))
    umap = pickle.load(open(f'{adir}/umap.pkl', 'rb'))
    sizes = np.bincount(labels, minlength=NUM_CLUSTERS)
    d = S['data']
    gt = np.load(f'{d["dir"]}/ground_truth.npz') if d['kind'] == 'demo' else None
    fig, axes = plt.subplots(1, 3 if gt is not None else 1, figsize=(15 if gt is not None else 6, 3.6), squeeze=False)
    axes = axes[0]
    axes[0].bar(range(NUM_CLUSTERS), sizes, color=plt.cm.tab20(np.arange(NUM_CLUSTERS) % 20))
    axes[0].set_xlabel('k-means cluster'); axes[0].set_ylabel('particles'); axes[0].set_xticks(range(NUM_CLUSTERS))
    if gt is not None:
        # the demo knows the truth: a good latent space separates the cap and orders the arm angle
        sc = axes[1].scatter(*umap.T, c=gt['arm_angle'], cmap='coolwarm', s=3)
        plt.colorbar(sc, ax=axes[1], label='true arm angle (°)')
        axes[2].scatter(*umap.T, c=np.where(gt['has_cap'], 'tab:green', 'tab:gray'), s=3)
        axes[2].set_title('green: cap present · grey: cap absent', fontsize=9)
        for ax in axes[1:]:
            ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
    plt.tight_layout(); plt.show()

_analyze()

## 6 · Reconstruct volumes

Runs `dsdsh eval_vol` to render maps at the k-means cluster centres (representative states, best quality) and along principal components (the main motions and compositional changes), then shows them. The `.mrc` files are in the analysis folder; in ChimeraX, `open reference*.mrc vseries true` plays a traversal as a movie.

In [ ]:
#@title 6.1 · K-means and PC-traversal maps { display-mode: "form" }
KMEANS_MAPS = True  #@param {type:"boolean"}
#@markdown Principal components to traverse, e.g. `1,2` (blank: none).
PC_TRAVERSALS = "1,2"  #@param {type:"string"}
#@markdown Pixel size of the maps in Å (0: the pixel size the model was trained at).
OUTPUT_APIX = 0  #@param {type:"number"}
FLIP_HANDEDNESS = False  #@param {type:"boolean"}
SHOW_3D = True  #@param {type:"boolean"}

def _volumes():
    a = need('analysis', '5.1')
    rd, N, K = S['runs'][a['run']]['dir'], a['epoch'], a['kmeans']
    if not os.path.exists(f'{rd}/weights.{N}.pkl'):
        raise RuntimeError(f'weights.{N}.pkl is gone (only the newest checkpoints are kept): analyze a newer epoch.')
    cfg = pickle.load(open(f'{rd}/config.pkl', 'rb'))
    apix = OUTPUT_APIX or round(float(cfg['model_args']['Apix']), 4)
    flip = '--flip' * FLIP_HANDEDNESS
    if KMEANS_MAPS:
        out = f'{a["dir"]}/kmeans{K}'
        sh(f'dsdsh eval_vol {rd} {N} kmeans {K} {apix} {flip}', quiet=True)
        maps = [read_mrc(f)[0] for f in mrc_series(out)]
        sizes = np.bincount(pickle.load(open(f'{out}/labels.pkl', 'rb')), minlength=K)
        print(f'{len(maps)} maps at {apix} Å/px in {out}')
        show_gallery([v.sum(0) for v in maps], [f'#{i} · {sizes[i]} ptcls' for i in range(len(maps))])
        if SHOW_3D:
            show_3d(maps, [f'k-means cluster {i} ({sizes[i]} particles)' for i in range(len(maps))], apix)
    for pc in [int(x) for x in re.findall(r'\d+', PC_TRAVERSALS)]:
        if pc > a['pcs']:
            print(f'PC{pc} was not sampled: raise NUM_PCS in 5.1.')
            continue
        out = f'{a["dir"]}/pc{pc}'
        sh(f'dsdsh eval_vol {rd} {N} pc {pc} {apix} {flip}', quiet=True)
        maps = [read_mrc(f)[0] for f in mrc_series(out)]
        print(f'PC{pc}: {len(maps)} maps in {out}')
        titles = [f'PC{pc} · step {i + 1} of {len(maps)}' for i in range(len(maps))]
        animate_views(maps, titles)
        if SHOW_3D:
            show_3d(maps, titles, apix)

_volumes()

In [ ]:
#@title 6.2 · *Optional* · Multi-body dynamics (OPUS-DSD2) { display-mode: "form" }
#@markdown For a run trained with MULTI_BODY: deform the map of one k-means cluster along a principal component of the dynamics latent space.
DYNAMICS_PC = 1  #@param {type:"integer"}
TEMPLATE_CLUSTER = 0  #@param {type:"integer"}
OUTPUT_APIX = 0  #@param {type:"number"}
SHOW_3D = True  #@param {type:"boolean"}

def _dynamics():
    a = need('analysis', '5.1')
    run = S['runs'][a['run']]
    if not run['arch'].get('multibody'):
        raise RuntimeError(f'Run "{a["run"]}" was trained without rigid bodies (MULTI_BODY in 4.1).')
    rd, N, K = run['dir'], a['epoch'], a['kmeans']
    cfg = pickle.load(open(f'{rd}/config.pkl', 'rb'))
    apix = OUTPUT_APIX or round(float(cfg['model_args']['Apix']), 4)
    sh(f'dsdsh eval_vol {rd} {N} dpc {DYNAMICS_PC} {apix} --masks {S["prep"]["mask_params"]} '
       f'--kmeans {K} --dfk {TEMPLATE_CLUSTER}', quiet=True)
    out = f'{rd}/defanalyze.{N}/pc{DYNAMICS_PC}'
    maps = [read_mrc(f)[0] for f in mrc_series(out)]
    print(f'{len(maps)} maps in {out}')
    titles = [f'cluster {TEMPLATE_CLUSTER} along dynamics PC{DYNAMICS_PC} · step {i + 1}' for i in range(len(maps))]
    animate_views(maps, titles)
    if SHOW_3D:
        show_3d(maps, titles, apix)

_dynamics()

In [ ]:
#@title 6.3 · 3D viewer for any map or folder of maps { display-mode: "form" }
#@markdown A `.mrc` file, or a folder of `reference*.mrc` maps (blank: the k-means maps of the last analysis). ISO_LEVEL 0 picks a level automatically.
MAP_PATH = ""  #@param {type:"string"}
ISO_LEVEL = 0  #@param {type:"number"}

def _viewer():
    path = MAP_PATH
    if not path:
        a = need('analysis', '5.1')
        path = f'{a["dir"]}/kmeans{a["kmeans"]}'
    files = mrc_series(path) if os.path.isdir(path) else [path]
    if not files:
        raise FileNotFoundError(f'no maps in {path} (run 6.1 first?)')
    maps = [read_mrc(f) for f in files]
    show_3d([m for m, _ in maps], [os.path.basename(f) for f in files], maps[0][1], level=ISO_LEVEL)

_viewer()

## 7 · Export

In [ ]:
#@title 7.1 · Particle subsets as STAR files { display-mode: "form" }
#@markdown Writes one STAR file per k-means cluster, made of the rows of your original STAR file so that it goes straight back into RELION, plus one with the clusters listed here, e.g. `0,3,5` (blank: skip).
SELECT_CLUSTERS = ""  #@param {type:"string"}

def _export():
    a = need('analysis', '5.1')
    d = S['data']
    K = a['kmeans']
    labels = pickle.load(open(f'{a["dir"]}/kmeans{K}/labels.pkl', 'rb'))
    src = cstar.Starfile.load(d['source_star'], relion31=d.get('source_relion31', d['relion31']))
    rows = np.load(d['indices']) if d.get('indices') else np.arange(len(src.df))
    if len(rows) != len(labels):
        raise RuntimeError(f'{len(labels)} labels but {len(rows)} particles: was the analysis made on other data?')
    out = f'{PROJECT_DIR}/exports/{a["run"]}_epoch{a["epoch"]}_kmeans{K}'
    os.makedirs(out, exist_ok=True)
    def write(name, keep):
        chosen = np.zeros(len(src.df), bool)
        chosen[rows[keep]] = True
        src.write_subset(f'{out}/{name}', chosen)
        return chosen.sum()
    for k in range(K):
        print(f'cluster{k:02d}.star: {write(f"cluster{k:02d}.star", labels == k)} particles')
    picked = [int(x) for x in re.findall(r'\d+', SELECT_CLUSTERS)]
    if picked:
        n = write('selected.star', np.isin(labels, picked))
        print(f'selected.star: {n} particles from clusters {picked}')
    print(f'\nWritten to {out}')

_export()

In [ ]:
#@title 7.2 · Download the results as a zip { display-mode: "form" }
#@markdown Analysis, maps, exported STAR files and training logs (everything is also in the project folder). The model weights add about 0.8 GB.
INCLUDE_WEIGHTS = False  #@param {type:"boolean"}

def _zip():
    import zipfile
    a = need('analysis', '5.1')
    rd, N = S['runs'][a['run']]['dir'], a['epoch']
    files = [f'{rd}/{x}' for x in ('config.pkl', 'run.log', 'train_cmd.txt', 'train_losses.png', f'z.{N}.pkl')]
    files += [f for f in glob.glob(f'{a["dir"]}/**/*', recursive=True) + glob.glob(f'{rd}/defanalyze.{N}/**/*', recursive=True)
              if os.path.isfile(f) and not f.endswith('cryoDRGN_filtering.ipynb')]
    files += glob.glob(f'{PROJECT_DIR}/exports/{a["run"]}_epoch{N}_*/*.star')
    if INCLUDE_WEIGHTS:
        files.append(f'{rd}/weights.{N}.pkl')
    zpath = f'{LOCAL_DIR}/opusdsd_{a["run"]}_epoch{N}.zip'
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in files:
            if os.path.exists(f):
                z.write(f, os.path.relpath(f, PROJECT_DIR))
    print(f'{zpath}: {os.path.getsize(zpath) / 2**20:.1f} MB')
    if IN_COLAB:
        from google.colab import files as colab_files
        colab_files.download(zpath)

_zip()

## Troubleshooting

* **CUDA out of memory**: lower `BATCH_SIZE` in 4.1 (then `TEMPLATERES` or `DOWNFRAC`) and start a new run.
* **The session disconnected**: re-run step 1 (and 2B if the particles were copied to the local disk). Re-running 4.1 with `RESUME` ticked continues training; analysis and volumes can be redone from any kept checkpoint.
* **`Uneven # columns` or other STAR parsing errors**: OPUS-DSD's parser expects one loop of particles without quoted or empty fields. Exporting the particles from RELION with `relion_star_handler` or saving them in RELION 3.0 format usually helps.
* **Particle stacks not found (2B)**: set `PARTICLES_ROOT` to the folder the `_rlnImageName` paths are relative to, typically the RELION project folder.
* **The UMAP is a thin ribbon**: `LAMBDA` was too high for the data; lower it and train again.
* **Inverted hand**: tick `FLIP_HANDEDNESS` in 6.1.

OPUS-DSD: Luo, Ni, Wang & Ma, *Nature Methods* 20, 1729–1738 (2023), [doi:10.1038/s41592-023-02031-6](https://www.nature.com/articles/s41592-023-02031-6). OPUS-DSD2: [bioRxiv 2024.09.24.614854](https://www.biorxiv.org/content/10.1101/2024.09.24.614854).